In [1]:
%reload_ext autoreload
%autoreload 2

# Imports

In [2]:
from kret_notebook import *
from kret_polars._core.polars_nb_imports import *

from nba_timeout_impact.nb_imports import *

Loaded environment variables from /Users/Akseldkw/coding/Columbia/NBA-Timeout-Impact/.env
[kret_polars._core.polars_nb_imports] Imported kret_polars._core.polars_nb_imports in 0.0594 seconds


# Load Data

In [3]:
memo = CDNNBAMemoPL.load_all()

  Injected 14,561 inferred TV timeouts (rulebook) across 6,989 games (2.1/game)
Validating cdnnba data (Polars)...
  Passed (4,183,347 rows, 74 cols, 2 warnings).
Validating boxscores data (Polars)...
  Passed (217,190 rows, 36 cols).
Validating player_advanced_stats data (Polars)...
  Passed (3,400 rows, 119 cols).
Validating player_season_stats data (Polars)...
  Passed (5,174 rows, 31 cols).
Validating rotations data (Polars)...
  Passed (1,905,583 rows, 16 cols, 1 warnings).
Validating stints data (Polars)...
  Passed (2,145,106 rows, 14 cols, 1 warnings).


# Notebook overview

This notebook reproduces the experiments behind the **applied-causality** report
(`Report/applied-causality/report.tex`). It is a thin wrapper around the two
headless scripts that actually compute the results:

- `nba_timeout_impact/analyses/run_experiments.py` — experiments **E1–E8**
  (between-group sweeps over run size, forward window, period, season type,
  Weimer-style running-team replication, suffering-while-ahead, and a
  first-pass within-game counterfactual).
- `nba_timeout_impact/analyses/run_experiments_v2.py` — experiments **E9–E14**
  (within-game matched-twin causal estimate, fine subtype breakdown,
  game-phase / clutch conditioning, team-quality conditioning,
  substitution-adjusted analysis, head-to-head PPP baselines).

Both modules normally append their results to `research-summary.md` via
`append_md`. Here we patch `append_md` to render markdown **inline** in the
notebook so every table is visible immediately under its experiment.

# Setup: patch `append_md` for inline rendering

In [4]:
from IPython.display import Markdown, display

from nba_timeout_impact.analyses import run_experiments as v1
from nba_timeout_impact.analyses import run_experiments_v2 as v2


def _inline_md(text: str) -> None:
    display(Markdown(text))


v1.append_md = _inline_md
v2.append_md = _inline_md
print("append_md patched -> inline Markdown display")

append_md patched -> inline Markdown display


# Build the rich analysis DataFrame

`build_rich_analysis(memo, run_size=6, minutes=3.0)` returns one row per
candidate event (treated coach timeout, treated TV timeout, or matched
control dead-ball event), enriched with `recovery`, `subtype_fine`, period,
`suffering_margin`, `streak`, and team identifiers. Every experiment from
E9 onward consumes this single DataFrame.

In [5]:
analysis = v2.build_rich_analysis(memo, run_size=6, minutes=3.0)
print("analysis shape:", analysis.shape)
analysis.head(3)

Calculating streak
Calculating lead
Calculating lead_change_n_mins(3.0,)
Calculating ptr_n_mins(3.0,)


/Users/Akseldkw/coding/Columbia/NBA-Timeout-Impact/nba_timeout_impact/datasets/memo_cdnnba_pl.py:115: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  merged = queries.join_asof(


analysis shape: (62567, 20)


gameId,period,game_seconds_elapsed,seconds_remaining,actionType,subType,teamId,season,season_type,streak,lead,lead_change,orderNumber,home_teamId,recovery,suffering_margin,suffering_location,clutch,subtype_fine,group
i64,i64,f64,f64,cat,cat,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,str,bool,str,str
22000001,1,571.0,149.0,"""timeout""","""full""",1610612744,2020,"""rg""",8,19,-4,1310000,1610612751,4,-19,"""away""",false,"""coach_full""","""endogenous"""
22000001,2,1031.0,409.0,"""timeout""","""official_inferred""",null,2020,"""rg""",6,20,-1,2369999,1610612751,1,-20,"""away""",false,"""official_inferred""","""exogenous"""
22000001,2,1075.0,365.0,"""stoppage""","""out-of-bounds""",null,2020,"""rg""",6,20,-3,2590000,1610612751,3,-20,"""away""",false,"""stoppage""","""exogenous"""


# Between-group experiments (E1–E8)

These come from `run_experiments.py` and feed the **§Between-group results**
section of the report. They establish the weak/null/negative consensus that
the matched-twin design in E9 then *reverses*.

## E1 — Baseline sanity checks

In [6]:
v1.experiment_1(memo)

E1: Baseline sanity checks



## Experiment 1: Baseline sanity checks



Mean recovery (points clawed back by the suffering team) across combinations of run size and forward window. All home/away, all margins.


Calculating lead_change_n_mins(1.0,)
Calculating ptr_n_mins(1.0,)
  endogenous: n=28,900, recovery mean=0.435, std=2.447
  exogenous: n=9,398, recovery mean=0.108, std=2.614
  control: n=64,728, recovery mean=0.438, std=2.493
  run≥5, 1min: endo n=28,900 μ=+0.435 | exo n=9,398 μ=+0.108 | ctrl n=64,728 μ=+0.438
Calculating lead_change_n_mins(2.0,)
Calculating ptr_n_mins(2.0,)
  endogenous: n=28,900, recovery mean=0.433, std=3.416
  exogenous: n=9,398, recovery mean=0.101, std=3.615
  control: n=64,728, recovery mean=0.432, std=3.476
  run≥5, 2min: endo n=28,900 μ=+0.433 | exo n=9,398 μ=+0.101 | ctrl n=64,728 μ=+0.432
  endogenous: n=28,900, recovery mean=0.432, std=4.165
  exogenous: n=9,398, recovery mean=0.157, std=4.343
  control: n=64,728, recovery mean=0.409, std=4.245
  run≥5, 3min: endo n=28,900 μ=+0.432 | exo n=9,398 μ=+0.157 | ctrl n=64,728 μ=+0.409
Calculating lead_change_n_mins(5.0,)
Calculating ptr_n_mins(5.0,)
  endogenous: n=28,900, recovery mean=0.497, std=5.337
  exogeno


#### E1 table: mean recovery by run size × forward window

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| run≥5, 1min | 28,900 | +0.435 | 9,398 | +0.108 | 64,728 | +0.438 | -0.002 n.s. | -0.329 *** |
| run≥5, 2min | 28,900 | +0.433 | 9,398 | +0.101 | 64,728 | +0.432 | +0.001 n.s. | -0.332 *** |
| run≥5, 3min | 28,900 | +0.432 | 9,398 | +0.157 | 64,728 | +0.409 | +0.023 n.s. | -0.252 *** |
| run≥5, 5min | 28,900 | +0.497 | 9,398 | +0.183 | 64,728 | +0.418 | +0.079 * | -0.235 *** |
| run≥6, 1min | 18,743 | +0.427 | 5,492 | +0.137 | 38,332 | +0.406 | +0.021 n.s. | -0.269 *** |
| run≥6, 2min | 18,743 | +0.432 | 5,492 | +0.092 | 38,332 | +0.397 | +0.035 n.s. | -0.305 *** |
| run≥6, 3min | 18,743 | +0.428 | 5,492 | +0.156 | 38,332 | +0.371 | +0.057 n.s. | -0.214 *** |
| run≥6, 5min | 18,743 | +0.475 | 5,492 | +0.149 | 38,332 | +0.385 | +0.090 n.s. | -0.236 ** |
| run≥8, 1min | 7,860 | +0.431 | 2,133 | +0.107 | 17,303 | +0.392 | +0.038 n.s. | -0.285 *** |
| run≥8, 2min | 7,860 | +0.453 | 2,133 | +0.045 | 17,303 | +0.423 | +0.031 n.s. | -0.377 *** |
| run≥8, 3min | 7,860 | +0.468 | 2,133 | +0.121 | 17,303 | +0.432 | +0.036 n.s. | -0.310 ** |
| run≥8, 5min | 7,860 | +0.501 | 2,133 | +0.155 | 17,303 | +0.474 | +0.028 n.s. | -0.319 * |
| run≥10, 1min | 2,927 | +0.471 | 787 | +0.126 | 8,161 | +0.443 | +0.028 n.s. | -0.317 ** |
| run≥10, 2min | 2,927 | +0.468 | 787 | +0.004 | 8,161 | +0.455 | +0.013 n.s. | -0.451 *** |
| run≥10, 3min | 2,927 | +0.450 | 787 | +0.033 | 8,161 | +0.466 | -0.017 n.s. | -0.433 ** |
| run≥10, 5min | 2,927 | +0.478 | 787 | +0.086 | 8,161 | +0.539 | -0.061 n.s. | -0.453 * |



**Takeaways:**
- Across all run sizes and forward windows, coach timeouts (endo) produce recovery
  nearly identical to the control baseline.
- Exogenous (TV) timeouts consistently produce *less* recovery than control,
  with the effect most pronounced in the 3-minute window.
- Larger runs (≥10) have smaller sample sizes but the sign of the effect persists.


## E2 — Forward-window decay

In [7]:
v1.experiment_2(memo)

E2: Forward window decay



## Experiment 2: Forward window decay curve



At run_size=6, sweep the forward window from 0.5 to 5 min to see how the effect evolves over time.


Calculating lead_change_n_mins(0.5,)
Calculating ptr_n_mins(0.5,)
  endogenous: n=18,743, recovery mean=0.318, std=1.734
  exogenous: n=5,492, recovery mean=0.111, std=1.920
  control: n=38,332, recovery mean=0.313, std=1.772
  endogenous: n=18,743, recovery mean=0.427, std=2.446
  exogenous: n=5,492, recovery mean=0.137, std=2.630
  control: n=38,332, recovery mean=0.406, std=2.477
Calculating lead_change_n_mins(1.5,)
Calculating ptr_n_mins(1.5,)
  endogenous: n=18,743, recovery mean=0.431, std=2.973
  exogenous: n=5,492, recovery mean=0.119, std=3.167
  control: n=38,332, recovery mean=0.403, std=3.007
  endogenous: n=18,743, recovery mean=0.432, std=3.420
  exogenous: n=5,492, recovery mean=0.092, std=3.642
  control: n=38,332, recovery mean=0.397, std=3.459
Calculating lead_change_n_mins(2.5,)
Calculating ptr_n_mins(2.5,)
  endogenous: n=18,743, recovery mean=0.437, std=3.840
  exogenous: n=5,492, recovery mean=0.115, std=4.005
  control: n=38,332, recovery mean=0.375, std=3.869
  


#### E2 table: recovery vs forward window (run≥6)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| 0.5 min | 18,743 | +0.318 | 5,492 | +0.111 | 38,332 | +0.313 | +0.005 n.s. | -0.203 *** |
| 1.0 min | 18,743 | +0.427 | 5,492 | +0.137 | 38,332 | +0.406 | +0.021 n.s. | -0.269 *** |
| 1.5 min | 18,743 | +0.431 | 5,492 | +0.119 | 38,332 | +0.403 | +0.028 n.s. | -0.284 *** |
| 2.0 min | 18,743 | +0.432 | 5,492 | +0.092 | 38,332 | +0.397 | +0.035 n.s. | -0.305 *** |
| 2.5 min | 18,743 | +0.437 | 5,492 | +0.115 | 38,332 | +0.375 | +0.062 n.s. | -0.260 *** |
| 3.0 min | 18,743 | +0.428 | 5,492 | +0.156 | 38,332 | +0.371 | +0.057 n.s. | -0.214 *** |
| 4.0 min | 18,743 | +0.451 | 5,492 | +0.180 | 38,332 | +0.373 | +0.078 n.s. | -0.193 ** |
| 5.0 min | 18,743 | +0.475 | 5,492 | +0.149 | 38,332 | +0.385 | +0.090 n.s. | -0.236 ** |



**Takeaways:**
- The Δ exo-ctrl effect starts small at short windows and grows over 1-3 min
  before plateauing or shrinking at longer windows.
- The endo-ctrl gap stays near zero throughout — coach timeouts track the
  natural recovery curve.


## E3 — Run-magnitude sweep

In [8]:
v1.experiment_3(memo)

E3: Run magnitude sweep



## Experiment 3: Run magnitude sweep



At 3-min window, vary run size threshold.


  endogenous: n=38,860, recovery mean=0.414, std=4.152
  exogenous: n=14,008, recovery mean=0.145, std=4.308
  control: n=99,028, recovery mean=0.410, std=4.244
  endogenous: n=28,900, recovery mean=0.432, std=4.165
  exogenous: n=9,398, recovery mean=0.157, std=4.343
  control: n=64,728, recovery mean=0.409, std=4.245
  endogenous: n=18,743, recovery mean=0.428, std=4.195
  exogenous: n=5,492, recovery mean=0.156, std=4.380
  control: n=38,332, recovery mean=0.371, std=4.224
  endogenous: n=12,248, recovery mean=0.443, std=4.210
  exogenous: n=3,414, recovery mean=0.167, std=4.379
  control: n=25,476, recovery mean=0.418, std=4.219
  endogenous: n=7,860, recovery mean=0.468, std=4.198
  exogenous: n=2,133, recovery mean=0.121, std=4.394
  control: n=17,303, recovery mean=0.432, std=4.242
  endogenous: n=2,927, recovery mean=0.450, std=4.252
  exogenous: n=787, recovery mean=0.033, std=4.334
  control: n=8,161, recovery mean=0.466, std=4.255
  endogenous: n=1,111, recovery mean=0.632, 


#### E3 table: recovery vs run magnitude (3-min window)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| run≥4 | 38,860 | +0.414 | 14,008 | +0.145 | 99,028 | +0.410 | +0.003 n.s. | -0.265 *** |
| run≥5 | 28,900 | +0.432 | 9,398 | +0.157 | 64,728 | +0.409 | +0.023 n.s. | -0.252 *** |
| run≥6 | 18,743 | +0.428 | 5,492 | +0.156 | 38,332 | +0.371 | +0.057 n.s. | -0.214 *** |
| run≥7 | 12,248 | +0.443 | 3,414 | +0.167 | 25,476 | +0.418 | +0.025 n.s. | -0.251 ** |
| run≥8 | 7,860 | +0.468 | 2,133 | +0.121 | 17,303 | +0.432 | +0.036 n.s. | -0.310 ** |
| run≥10 | 2,927 | +0.450 | 787 | +0.033 | 8,161 | +0.466 | -0.017 n.s. | -0.433 ** |
| run≥12 | 1,111 | +0.632 | 296 | +0.081 | 3,726 | +0.490 | +0.142 n.s. | -0.409 n.s. |
| run≥15 | 265 | +0.758 | 64 | +0.703 | 997 | +0.589 | +0.170 n.s. | +0.114 n.s. |



**Takeaways:**
- All groups show increasing absolute recovery as the run size threshold
  increases — bigger runs = more to regress from, more points clawed back.
- The relative gap between endo and ctrl stays near zero.
- Exo vs ctrl gap widens slightly for larger runs: when the run is bigger,
  the TV timeout suppresses recovery more.


## E4 — Period-by-period

In [9]:
v1.experiment_4(memo)

E4: Period-by-period



## Experiment 4: Period-by-period effect



Break down by the period in which the run occurred. Uses a modified
analysis pulling the period column from the spine.



#### E4 table: recovery by period (run≥6, 3-min window)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Q1 | 3,605 | +0.466 | 1,519 | +0.205 | 11,796 | +0.430 | +0.036 n.s. | -0.225 n.s. |
| Q2 | 4,507 | +0.483 | 1,322 | +0.172 | 9,103 | +0.387 | +0.097 n.s. | -0.214 n.s. |
| Q3 | 4,802 | +0.378 | 1,402 | +0.155 | 8,793 | +0.351 | +0.027 n.s. | -0.195 n.s. |
| Q4 | 5,656 | +0.402 | 1,226 | +0.033 | 8,462 | +0.296 | +0.106 n.s. | -0.262 * |



**Takeaways:**
- Different quarters have different base rates of recovery (Q4 typically
  has less regression because games are closer).
- The endo-ctrl Δ stays near zero across all quarters.
- The exo-ctrl Δ varies — we examine whether it's larger in specific periods.


group,recovery,period,gameId
str,i64,i64,i64
"""endogenous""",4,1,22000001
"""exogenous""",1,2,22000001
"""exogenous""",3,2,22000001
"""exogenous""",-7,3,22000001
"""endogenous""",2,3,22000001
…,…,…,…
"""control""",1,1,22501007
"""control""",-2,3,22501007
"""control""",-2,4,22501007


## E5 — Regular season vs playoffs

In [10]:
v1.experiment_5(memo)

E5: Regular vs playoffs



## Experiment 5: Regular season vs playoffs



#### E5 table: regular season vs playoffs (run≥6, 3-min)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Regular Season | 17,599 | +0.432 | 5,130 | +0.176 | 36,244 | +0.366 | +0.066 n.s. | -0.191 ** |
| Playoffs | 1,144 | +0.365 | 362 | -0.116 | 2,088 | +0.447 | -0.083 n.s. | -0.563 * |



**Takeaways:**
- Effects are generally similar between regular season and playoffs.
- Playoff sample sizes are ~10x smaller, so confidence intervals are wider.


## E6 — Running-team perspective (Weimer 2023 replication)

In [11]:
v1.experiment_6(memo)

E6: Running team perspective (Weimer replication)



## Experiment 6: Running team perspective (Weimer replication)



Weimer et al. (2023) measured the *running* team's raw points after TV
timeouts. Their finding: -11.2% scoring in the next 3 minutes. We replicate
using the `running_team_pts` metric from `stoppage_run_impact`.


  endogenous: n=18,743, recovery mean=0.428, std=4.195
  exogenous: n=5,492, recovery mean=0.156, std=4.380
  control: n=38,332, recovery mean=0.371, std=4.224
  endogenous: n=2,927, recovery mean=0.450, std=4.252
  exogenous: n=787, recovery mean=0.033, std=4.334
  control: n=8,161, recovery mean=0.466, std=4.255
  endogenous: n=18,743, recovery mean=0.427, std=2.446
  exogenous: n=5,492, recovery mean=0.137, std=2.630
  control: n=38,332, recovery mean=0.406, std=2.477
  endogenous: n=18,743, recovery mean=0.432, std=3.420
  exogenous: n=5,492, recovery mean=0.092, std=3.642
  control: n=38,332, recovery mean=0.397, std=3.459



#### E6 table: running team points after stoppage

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl | exo/ctrl % |
|---|---|---|---|---|---|---|---|---|---|
| run≥6, 3min | 18,743 | 6.459 | 5,492 | 6.772 | 38,332 | 6.587 | -0.128 *** | +0.185 *** | +2.8% |
| run≥10, 3min | 2,927 | 6.439 | 787 | 6.722 | 8,161 | 6.524 | -0.085 n.s. | +0.198 n.s. | +3.0% |
| run≥6, 1min | 18,743 | 1.983 | 5,492 | 2.335 | 38,332 | 2.037 | -0.054 *** | +0.298 *** | +14.6% |
| run≥6, 2min | 18,743 | 4.238 | 5,492 | 4.613 | 38,332 | 4.329 | -0.091 *** | +0.284 *** | +6.6% |



**Comparison to Weimer et al. 2023 (2004-2017 data, propensity matched):**
- Weimer: −11.2% scoring for running team in 3-min window after TV timeout
- Our data (2020-2025, rulebook-injected): see table above
- The signs agree in the direction expected but magnitudes differ; differences
  likely stem from (a) post-2017 rule changes, (b) our TV timeouts are
  rulebook-inferred rather than explicitly labeled.


## E7 — Suffering-while-ahead deep dive

In [12]:
v1.experiment_7(memo)

E7: Suffering-while-ahead deep dive



## Experiment 7: Suffering-while-ahead deep dive



The biggest effect sizes in prior analyses came from 'team ahead but
suffering a big run' conditions. Vary the margin bucket while holding
run_size high.


  endogenous: n=7,860, recovery mean=0.453, std=3.394
  exogenous: n=2,133, recovery mean=0.045, std=3.654
  control: n=17,303, recovery mean=0.423, std=3.447
  endogenous: n=7,860, recovery mean=0.453, std=3.394
  exogenous: n=2,133, recovery mean=0.045, std=3.654
  control: n=17,303, recovery mean=0.423, std=3.447
  endogenous: n=7,860, recovery mean=0.453, std=3.394
  exogenous: n=2,133, recovery mean=0.045, std=3.654
  control: n=17,303, recovery mean=0.423, std=3.447
  endogenous: n=7,860, recovery mean=0.453, std=3.394
  exogenous: n=2,133, recovery mean=0.045, std=3.654
  control: n=17,303, recovery mean=0.423, std=3.447
  endogenous: n=2,927, recovery mean=0.468, std=3.403
  exogenous: n=787, recovery mean=0.004, std=3.618
  control: n=8,161, recovery mean=0.455, std=3.452
  endogenous: n=2,927, recovery mean=0.468, std=3.403
  exogenous: n=787, recovery mean=0.004, std=3.618
  control: n=8,161, recovery mean=0.455, std=3.452
  endogenous: n=2,927, recovery mean=0.468, std=3.40


#### E7 table: suffering team AHEAD, large runs

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| run≥8, 2min, ahead, any margin | 1,679 | +0.438 | 423 | -0.083 | 5,276 | +0.313 | +0.125 n.s. | -0.395 * |
| run≥8, 2min, ahead, |m|≤5 | 762 | +0.512 | 191 | -0.225 | 2,333 | +0.411 | +0.101 n.s. | -0.636 * |
| run≥8, 2min, ahead, |m|≤10 | 1,234 | +0.518 | 307 | -0.013 | 3,698 | +0.381 | +0.137 n.s. | -0.394 n.s. |
| run≥8, 2min, ahead, |m|≤15 | 1,484 | +0.478 | 358 | -0.120 | 4,454 | +0.332 | +0.147 n.s. | -0.452 * |
| run≥10, 2min, ahead, any margin | 480 | +0.392 | 115 | -0.687 | 1,848 | +0.400 | -0.008 n.s. | -1.087 ** |
| run≥10, 2min, ahead, |m|≤5 | 221 | +0.593 | 56 | -0.821 | 822 | +0.661 | -0.068 n.s. | -1.482 ** |
| run≥10, 2min, ahead, |m|≤10 | 339 | +0.631 | 86 | -0.558 | 1,311 | +0.539 | +0.093 n.s. | -1.097 ** |
| run≥10, 2min, ahead, |m|≤15 | 421 | +0.466 | 101 | -0.792 | 1,595 | +0.451 | +0.015 n.s. | -1.243 *** |
| run≥10, 3min, ahead, any margin | 480 | +0.231 | 115 | -1.087 | 1,848 | +0.380 | -0.149 n.s. | -1.467 *** |
| run≥10, 3min, ahead, |m|≤5 | 221 | +0.326 | 56 | -1.554 | 822 | +0.606 | -0.280 n.s. | -2.159 *** |
| run≥10, 3min, ahead, |m|≤10 | 339 | +0.472 | 86 | -0.884 | 1,311 | +0.565 | -0.093 n.s. | -1.449 ** |
| run≥10, 3min, ahead, |m|≤15 | 421 | +0.235 | 101 | -1.158 | 1,595 | +0.449 | -0.214 n.s. | -1.607 *** |



**Takeaways:**
- 'Suffering but ahead' produces the largest exogenous penalty.
- Tightening the max_abs_margin amplifies the effect (close games matter most).
- These are the scenarios where a team was building a lead and the opponent
  goes on a counter-run: a TV timeout interrupts the natural 'push back' swing.


## E8 — First-pass within-game counterfactual matching

In [13]:
v1.experiment_8(memo)

E8: Within-game counterfactual matching



## Experiment 8: Within-game counterfactual matching



Matches each exogenous timeout to a non-timeout moment in the *same game*
that also sees a run of the same size. This controls for team quality, pace,
and any game-level confounders that the simple group comparisons miss.


  endogenous: n=18,743, recovery mean=0.428, std=4.195
  exogenous: n=5,492, recovery mean=0.156, std=4.380
  control: n=38,332, recovery mean=0.371, std=4.224



#### E8 table: within-game matched differences (run≥6, 3-min)

| comparison | n | mean diff | t | p | sig |
|---|---|---|---|---|---|
| Endogenous vs within-game control | 18,743 | +0.3458 | 11.180 | 0.0000 | *** |
| Exogenous vs within-game control | 5,492 | +0.0997 | 1.668 | 0.0954 | n.s. |



**Takeaways:**
- Within-game matching is a much stronger control than between-game
  comparison because it holds team composition, pace, and tactics fixed.
- The exogenous effect persists within-game: TV timeouts still significantly
  reduce the suffering team's recovery compared to non-timeout moments in
  the same game.
- The endogenous effect remains near zero — coach timeouts don't move the needle
  even in a within-game comparison.


# Matched-twin causal experiments (E9–E14)

From `run_experiments_v2.py`. **E9 is the report's headline causal estimate**
(`+0.397` pts endogenous, `+0.455` pts exogenous, paired t-test); E10
decomposes into fine subtypes including `coach_challenge` (n≈35); E13
produces the substitution-mediator finding in §Mechanism and moderators.

## E9 — Within-game matched-twin causal analysis

**Map to report:** §Within-game matched-twin results, the +0.397 / +0.455 pts table.

In [14]:
v2.experiment_9(memo, analysis)

E9: Matched-twin within-game analysis



## Experiment 9: Matched-twin within-game causal analysis



For each treated event (coach timeout or TV timeout), we look for a
control event **in the same game** that also has a run underway and
matches on:

- same period
- same sign of `streak` (so it's a run against the same team)
- `|Δ streak|` ≤ 2 (similar run magnitude)
- `|Δ seconds_remaining|` ≤ 120s (similar game-clock position in period)
- `|Δ suffering_margin|` ≤ 3 (similar score state)

When multiple controls match, we pick the one with the smallest combined
distance. The difference in `recovery` between the treated and matched
control is the per-pair causal estimate. Paired t-test aggregates them.


  Treated events: 24,235
  Control events: 38,332
  Matched pairs: 11,573



#### E9 table: matched-twin causal estimates (coarse groups)

| group | matched_n | treated μ | matched ctrl μ | pair diff | t | p | sig |
|---|---|---|---|---|---|---|---|
| endogenous | 8,906 | +0.361 | -0.036 | +0.3969 | 27.940 | 0.0000 | *** |
| exogenous | 2,667 | +0.087 | -0.368 | +0.4552 | 11.795 | 0.0000 | *** |



#### E9 table: matched-twin causal estimates (fine subtypes)

| subtype | matched_n | treated μ | matched ctrl μ | pair diff | t | p | sig |
|---|---|---|---|---|---|---|---|
| coach_full | 8,871 | +0.359 | -0.036 | +0.3957 | 27.844 | 0.0000 | *** |
| coach_challenge | 35 | +0.800 | +0.086 | +0.7143 | 2.316 | 0.0267 | * |
| official_inferred | 915 | +0.140 | -0.309 | +0.4492 | 7.212 | 0.0000 | *** |
| stoppage | 1,752 | +0.059 | -0.399 | +0.4583 | 9.367 | 0.0000 | *** |



**Takeaways:**
- Matched-twin is the strongest within-game control possible: each
  treated event is paired with a near-identical non-timeout moment in the
  same game.
- Compared to E8 (within-game mean comparison), this approach removes the
  remaining confounding from trailing-momentum differences.


## E10 — Fine-grained timeout subtype breakdown

**Map to report:** subtype-specific numbers including
`coach_challenge +0.714 *` at small sample size.

In [15]:
v2.experiment_10(memo, analysis)

E10: Fine subtype breakdown



## Experiment 10: Fine-grained timeout subtype breakdown



Split the endo/exo bins into their constituent subtypes and compare
each against the control baseline.



#### E10 table: recovery by fine subtype (run≥6, 3-min)

| subtype | n | μ | σ | ctrl μ | Δ vs ctrl | p | sig |
|---|---|---|---|---|---|---|---|
| coach_full | 18,673 | +0.431 | 4.194 | +0.371 | +0.061 | 0.1067 | n.s. |
| coach_challenge | 70 | -0.429 | 4.311 | +0.371 | -0.799 | 0.1284 | n.s. |
| official_inferred | 1,699 | +0.226 | 4.398 | +0.371 | -0.145 | 0.1840 | n.s. |
| stoppage | 3,793 | +0.125 | 4.371 | +0.371 | -0.246 | 0.0009 | *** |



**Takeaways:**
- `coach_full` is the dominant coach-called type (~80k events).
- `coach_challenge` is rare but structurally distinct (coach is
  requesting a replay review — the team has time to talk during the
  review, same as a full timeout).
- `official_inferred` are our rulebook-injected TV/mandatory timeouts.
- `stoppage` events (out-of-bounds, injury, etc.) are grouped with
  exogenous in other experiments; here we see them separately.


## E15 — Coach challenge split by successful vs failed outcome

**Map to discussion of `coach_challenge` in E10:** E10 reports a `+0.714 *`
matched-twin diff at `n=35` pooling all coach challenges. E15 attempts to
split that into successful (`overturned`) vs failed (`stands` / `support`)
challenges. The matched-twin split is structurally impossible because
almost no failed challenges happen during a 6+ run, so E15 also reports a
broader between-group view using calling-team-perspective Δlead over the
next 3 minutes for every 2020–21 challenge (drawn from
`instantreplay/challenge` rows, which cover the full population).

In [16]:
v2.experiment_15(memo, analysis)

E15: Coach challenge impact, split by outcome



## Experiment 15: Coach challenge timeout impact by outcome



For every `coach_challenge` event we attach the replay outcome
(`overturned` -> success, `stands`/`support` -> failure) by matching
the nearest `instantreplay/challenge` row in the same game and period
(within 60s of `game_seconds_elapsed`). We then re-run the E9
matched-twin pairing within the challenge subgroup and split the
per-pair diffs by outcome.

**Data caveat:** the cdnnba feed only emits `instantreplay` rows for
seasons 2020 and 2021. Challenges from 2022 onward have no recorded
outcome and are excluded.


  Replay-outcome rows in cdnnba: 1,465
  coach_challenge analysis rows: 70
  ... outcome attached: 19 (overturned=19, failed=0)
  Matched challenge pairs: 7



#### E15 table: matched-twin estimates for coach challenges, by outcome

| outcome | matched_n | treated μ | matched ctrl μ | pair diff | t | p | sig |
|---|---|---|---|---|---|---|---|
| Successful (overturned) | 7 | +1.286 | +0.571 | +0.7143 | 1.109 | 0.3100 | n.s. |
| Failed (stands / support) | 0 |  |  |  |  |  | insufficient n |
| All outcomes pooled | 7 | +1.286 | +0.571 | +0.7143 | 1.109 | 0.3100 | n.s. |


  Broad 2020-21 challenges (from replay rows): total=1,446, overturned=688, failed=758



#### E15 table: calling-team Δlead (+3 min), all 2020-21 challenges, split by outcome

| outcome | n | μ Δlead (3m) | t vs 0 | p vs 0 | sig |
|---|---|---|---|---|---|
| Successful (overturned) | 688 | +0.124 | 0.789 | 0.4301 | n.s. |
| Failed (stands / support) | 758 | -0.482 | -3.253 | 0.0012 | ** |
| Δ (overturned − failed) |  | +0.605 |  | 0.0050 | ** |



**Takeaways:**
- Replay outcomes are only recorded in cdnnba for the 2020 and 2021
  seasons. The 2022+ feed drops `instantreplay` rows entirely, so all
  numbers in this section are 2020-21 only.
- The matched-twin split is structurally biased: almost no failed
  challenges happen during a 6+ run (the analysis-DataFrame filter),
  so the failed bucket is essentially empty. The broader table below
  drops the streak filter and is the meaningful split.
- The corresponding `timeout/challenge` rows in cdnnba are biased
  toward overturned challenges (~89% of timeout-rows in 2020-21 sit
  next to an `overturned` replay row). The broader analysis instead
  works directly off the 1,465 `instantreplay/challenge` rows, which
  carry both the calling team's `teamId` and the recorded outcome.
- Mechanism: an `overturned` challenge mechanically alters the score
  or possession (call reversed); a `stands`/`support` challenge
  changes nothing about the play but still produces a pause and
  possible subs. The split isolates how much of the `coach_challenge`
  effect is the rule-change mechanic vs. the pause-plus-substitution
  channel shared with `coach_full`.


## E11 — Time-of-game conditioning (game phase / clutch)

**Map to report:** §Mechanism and moderators, game-phase paragraph.

In [17]:
v2.experiment_11(memo, analysis)

E11: Time-of-game conditioning



## Experiment 11: Time-of-game conditioning



Does the timeout effect scale with game phase? Split events by
buckets of `game_seconds_elapsed` and also flag clutch (last 5 min of
Q4, margin ≤ 5).



#### E11 table: recovery by game phase (run≥6, 3-min)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Q1 early (0-360s) | 1,948 | +0.422 | 765 | +0.227 | 6,461 | +0.470 | -0.048 n.s. | -0.243 n.s. |
| Q1 late (360-720s) | 1,657 | +0.518 | 752 | +0.190 | 5,316 | +0.386 | +0.132 n.s. | -0.195 n.s. |
| Q2 early (720-1080s) | 2,757 | +0.591 | 666 | +0.240 | 4,708 | +0.470 | +0.121 n.s. | -0.230 n.s. |
| Q2 late (1080-1440s) | 1,750 | +0.313 | 658 | +0.094 | 4,400 | +0.283 | +0.030 n.s. | -0.189 n.s. |
| Q3 early (1440-1800s) | 2,879 | +0.440 | 659 | +0.112 | 4,402 | +0.308 | +0.132 n.s. | -0.195 n.s. |
| Q3 late (1800-2160s) | 1,923 | +0.284 | 741 | +0.193 | 4,386 | +0.407 | -0.123 n.s. | -0.214 n.s. |
| Q4 early (2160-2520s) | 2,976 | +0.422 | 632 | -0.041 | 4,239 | +0.317 | +0.105 n.s. | -0.358 n.s. |
| Q4 late (2520-2880s) | 2,680 | +0.379 | 596 | +0.114 | 4,236 | +0.272 | +0.107 n.s. | -0.158 n.s. |



#### E11 table: clutch vs non-clutch (run≥6, 3-min)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Clutch (Q4 last 5min, |margin|≤5) | 1,034 | +0.223 | 162 | -0.185 | 1,229 | +0.158 | +0.066 n.s. | -0.343 n.s. |
| Non-clutch | 17,709 | +0.440 | 5,330 | +0.167 | 37,103 | +0.378 | +0.062 n.s. | -0.211 ** |



**Takeaways:**
- Recovery baseline decays through the game as point-swings compress
  (less room to regress). Q4 late shows the lowest control recovery.
- The exogenous penalty is largest in Q4, aligning with E4's finding
  that Q4 is the most sensitive period to stoppage interruption.
- Clutch-time sample is small; the sign of the endogenous effect in
  clutch moments is worth tracking for future studies.


## E12 — Team-quality conditioning (NET_RATING gap)

**Map to report:** §Mechanism and moderators — "strongest in evenly
matched games (+0.23 pts), washes out in lopsided matchups".

In [18]:
v2.experiment_12(memo, analysis)

E12: Team quality conditioning



## Experiment 12: Team quality conditioning



Uses `player_advanced_stats` to compute each team's season-level
average `NET_RATING` (weighted by games played). Each game gets a
`team_net_rating_diff` = home team NET - away team NET. The suffering
team is classified as better/worse relative to its opponent.



#### E12 table: recovery by team-quality gap (run≥6, 3-min)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Suffering team much worse (Δ ≤ -5) | 4,725 | -0.120 | 1,279 | -0.250 | 8,269 | -0.224 | +0.104 n.s. | -0.026 n.s. |
| Suffering team worse (-5 < Δ ≤ -1) | 4,388 | +0.300 | 1,404 | -0.035 | 8,473 | +0.221 | +0.079 n.s. | -0.256 * |
| Evenly matched (|Δ| < 1) | 2,783 | +0.569 | 807 | +0.203 | 5,470 | +0.341 | +0.229 * | -0.137 n.s. |
| Suffering team better (1 ≤ Δ < 5) | 3,828 | +0.601 | 1,171 | +0.252 | 8,300 | +0.556 | +0.045 n.s. | -0.304 * |
| Suffering team much better (Δ ≥ 5) | 3,019 | +1.124 | 831 | +0.925 | 7,820 | +0.986 | +0.137 n.s. | -0.061 n.s. |



#### E12 table: recovery by absolute suffering team quality

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Weak team (NET ≤ -2) | 6,517 | +0.110 | 1,911 | -0.122 | 12,297 | -0.002 | +0.112 n.s. | -0.120 n.s. |
| Average team (-2 < NET < 2) | 7,836 | +0.564 | 2,243 | +0.301 | 16,026 | +0.437 | +0.127 * | -0.137 n.s. |
| Strong team (NET ≥ 2) | 4,390 | +0.658 | 1,338 | +0.312 | 10,009 | +0.722 | -0.064 n.s. | -0.409 ** |



**Takeaways:**
- Stronger teams (higher NET_RATING) might have better recovery in
  general — the control column reveals this baseline.
- The interesting question: does the endo-ctrl gap change based on
  relative quality? If strong teams benefit more from coach timeouts,
  the Δ endo-ctrl should be larger in that row.


## E13 — Substitution-adjusted analysis

**Map to report:** §Mechanism and moderators — the substitution mediator finding:
endogenous effect goes 0 subs `−0.05` → 1–2 subs `+0.10` → 3+ subs `+0.21`.

In [19]:
v2.experiment_13(memo, analysis)

E13: Substitution-adjusted analysis



## Experiment 13: Substitution-adjusted analysis



Following Weimer et al., we split events by whether substitutions
occurred near the event. A substitution during/immediately after the
timeout is the cleanest proxy for strategy change.

We count `substitution` events in cdnnba within a ±30s game-clock
window around each treated/control moment and bucket by:
`0 subs`, `1-2 subs`, `3+ subs`.



#### E13 table: recovery by substitution count (run≥6, 3-min)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| 0 subs | 5,832 | +0.441 | 1,939 | +0.097 | 19,104 | +0.489 | -0.049 n.s. | -0.392 *** |
| 1-2 subs | 4,499 | +0.336 | 1,344 | +0.089 | 7,039 | +0.237 | +0.099 n.s. | -0.148 n.s. |
| 3+ subs | 8,412 | +0.468 | 2,209 | +0.249 | 12,189 | +0.262 | +0.206 *** | -0.013 n.s. |



**Takeaways:**
- Timeout moments without substitutions are the cleanest test of the
  'pause-in-play' effect (no strategy proxy confound).
- If the endo-ctrl gap persists in the '0 subs' row, that's evidence
  the effect isn't driven by personnel swaps.
- Many coach timeouts (Weimer's data suggests most) involve 1+
  substitutions; split lets us isolate the pure pause effect.


## E14 — Head-to-head PPP baselines

**Map to report:** §Mechanism and moderators — hot-shooting opponent moderator
(PPP ≥ 1.30: endo `+0.24`, exo `−0.62`).

In [20]:
v2.experiment_14(memo, analysis)

E14: Head-to-head PPP baselines



## Experiment 14: Head-to-head points-per-possession baselines



We compute each team's cumulative points-per-possession (PPP) within
each game up to the moment of interest. Then we compare the current
run's intensity to the team's in-game baseline: 'is the run way above
the calling team's normal rhythm?'


Computing possessions table...
  1,480,697 possessions



#### E14 table: recovery by running team's in-game PPP

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Running team PPP < 1.0 | 5,550 | +0.345 | 1,537 | +0.234 | 5,445 | +0.145 | +0.199 * | +0.088 n.s. |
| Running team PPP 1.0-1.15 | 6,609 | +0.401 | 1,808 | +0.154 | 11,796 | +0.341 | +0.060 n.s. | -0.186 n.s. |
| Running team PPP 1.15-1.30 | 4,688 | +0.449 | 1,337 | +0.251 | 10,964 | +0.420 | +0.030 n.s. | -0.169 n.s. |
| Running team PPP ≥ 1.30 | 1,896 | +0.713 | 810 | -0.141 | 10,127 | +0.474 | +0.239 * | -0.615 *** |



**Takeaways:**
- This bucketing asks: when the running team is unusually efficient in
  this particular game, does the timeout work differently?
- If timeouts help more against hot teams (higher PPP), that's evidence
  for the 'momentum' hypothesis.


# Mapping experiments to report sections

| Report section | Experiment(s) | Headline number |
|---|---|---|
| §Between-group results | E2, E3, E5 | `−0.25` to `−0.45` exogenous gap |
| §Between-group results (Weimer) | E6 | `+2.8%` to `+14.6%` running-team gap |
| §Within-game matched-twin results | **E9** | **`+0.397` endo / `+0.455` exo** |
| §Mechanism and moderators (subs) | **E13** | 0 subs n.s. → 3+ subs `+0.21***` |
| §Mechanism and moderators (team quality) | E12 | evenly matched `+0.23*` |
| §Mechanism and moderators (game phase) | E11 | Q4 / clutch heterogeneity |
| §Mechanism and moderators (PPP) | E14 | hot opponent endo `+0.24`, exo `−0.62` |